In [ ]:
# We need rank_bm25 for sparse retrieval
!pip install -q langchain langchain-groq langchain-community langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu gradio rank_bm25

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

In [ ]:
GROQ_API_KEY = "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"
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
loader = WebBaseLoader("https://www.rtx.com/collinsaerospace/what-we-do/capabilities")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

In [ ]:
vectorstore = FAISS.from_documents(splits, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
sparse_retriever = BM25Retriever.from_documents(splits)
sparse_retriever.k = 3

In [ ]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever], 
    weights=[0.5, 0.5]
)

In [ ]:
template = """You are an expert technical assistant. Use the pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 

Context: {context}

Question: {question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
def hybrid_chat(message, history):
    return chain.invoke(message)

demo = gr.ChatInterface(
    fn=hybrid_chat, 
    title="Llama 3.1 Advanced Hybrid RAG",
    description="Hybrid Search: Dense (FAISS) + Sparse (BM25) for better accuracy."
)

In [ ]:
if __name__ == "__main__":
    demo.launch()